# Data Science Task: QS World University Rankings 2025


---
## Section 1: Import Libraries

In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


---
## Section 2: Load Dataset

### Q1. Data Loading & Inspection

In [ ]:
data = pd.read_csv("QS_World_University_Rankings_2025_Top_global_universities.csv", encoding="latin-1")


In [ ]:
data.head(5)


In [ ]:
data.shape


In [ ]:
data.info()


In [ ]:
data.isnull().sum()


---
## Section 3: Data Cleaning

### Q2. Data Cleaning

**Data Cleaning Logic:**

- `Overall_Score` and `RANK_2025` are stored as strings (some have `=` suffixes like `15=`). We strip those and convert to numeric.
- Score columns already numeric; rank columns (string) get the same treatment.
- `Overall_Score` has ~902 missing values — these are universities ranked beyond 500 where QS does not publish a score. We impute with the **median** so we keep all rows.
- Other score columns with small gaps are also median-imputed.
- Categorical columns (`STATUS`, `RANK_2024`) with few nulls are filled with `'Unknown'` / forward-fill.


In [ ]:
print('Shape before cleaning:', data.shape)
print(data.isnull().sum())


In [ ]:
df = data.copy()

def strip_rank(val):
    """Remove trailing = or + from rank strings and return int, NaN if unparseable."""
    if pd.isna(val):
        return np.nan
    cleaned = str(val).strip().rstrip('=+').split('-')[0]
    try:
        return int(cleaned)
    except ValueError:
        return np.nan

rank_cols = [c for c in df.columns if 'Rank' in c or c.startswith('RANK')]
for col in rank_cols:
    df[col] = df[col].apply(strip_rank)

df['Overall_Score'] = pd.to_numeric(
    df['Overall_Score'].astype(str).str.rstrip('=+'), errors='coerce'
)


In [ ]:
score_cols = [c for c in df.columns if 'Score' in c]
for col in score_cols:
    med = df[col].median()
    df[col] = df[col].fillna(med)

df['STATUS'] = df['STATUS'].fillna('Unknown')
df['RANK_2024'] = df['RANK_2024'].fillna(method='ffill')

print('Shape after cleaning:', df.shape)
print(df.isnull().sum())


In [ ]:
df.head(5)


### Q3. Feature Engineering (Using NumPy)

**Score_Category rules:**
- Elite → Overall_Score ≥ 90
- High → 75–89
- Medium → 50–74
- Low → < 50


In [ ]:
s = df['Overall_Score'].values
cats = np.where(s >= 90, 'Elite',
        np.where(s >= 75, 'High',
        np.where(s >= 50, 'Medium', 'Low')))
df['Score_Category'] = cats
df[['Institution_Name', 'Overall_Score', 'Score_Category']].head(10)


In [ ]:
df['Score_Category'].value_counts()


---
## Section 4: EDA (Exploratory Data Analysis)

### Q4. Top Universities Analysis

In [ ]:
top10 = df.nsmallest(10, 'RANK_2025')[['Institution_Name', 'RANK_2025', 'Overall_Score', 'Location']]
top10


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
colors = sns.color_palette('Blues_r', 10)
bars = ax.barh(top10['Institution_Name'][::-1], top10['Overall_Score'][::-1], color=colors)
ax.set_xlabel('Overall Score', fontsize=12)
ax.set_title('Top 10 Universities by QS Rank 2025', fontsize=14, fontweight='bold')
for bar, score in zip(bars, top10['Overall_Score'][::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{score:.1f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()


**Insights:**

1. MIT holds the top spot with a perfect score of 100, maintaining its #1 position from 2024.
2. UK universities (Oxford, Cambridge, Imperial) dominate the top 5 alongside US institutions, showing Anglo-American dominance in global rankings.
3. The score gap between rank 1 (100) and rank 10 (Caltech, ~90.9) is only ~9 points, suggesting the elite tier is tightly clustered.


### Q5. Country-wise Dominance

In [ ]:
country_counts = df['Location'].value_counts().reset_index()
country_counts.columns = ['Country', 'University_Count']
top10_countries = country_counts.head(10)
top10_countries


In [ ]:
fig = px.bar(
    top10_countries,
    x='Country', y='University_Count',
    color='University_Count',
    color_continuous_scale='Blues',
    title='Top 10 Countries by Number of Ranked Universities (QS 2025)',
    labels={'University_Count': 'Number of Universities'},
    text='University_Count'
)
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_tickangle=-30)
fig.show()


**Interpretation:**

The United States leads by a wide margin with the most ranked universities, followed by the United Kingdom. Asian countries like China, Japan, and South Korea are strongly represented, reflecting the growing investment in higher education across Asia. Australia punches above its weight given its population size.
